In [ ]:
from torch import Tensor
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Install huggingface_hub if not already installed
!pip install huggingface_hub

# Use the token to log in
from huggingface_hub import login
login("")

In [ ]:
# Load model directly

base_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B").to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [ ]:
# Load model directly

layerskip_tokenizer = AutoTokenizer.from_pretrained("facebook/layerskip-llama3.2-1B")
layerskip_model = AutoModelForCausalLM.from_pretrained("facebook/layerskip-llama3.2-1B").to(device)

In [ ]:
# Test the text generation - Base Llama
prompt = "What is the meaning of life?"
inputs = base_tokenizer(prompt, return_tensors="pt").to(device)

outputs = base_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True
)
print(base_tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What is the meaning of life? This is a question that we are asked all the time and I don’t think I have the answer. It’s a question that is asked by the people that live in the past and it’s a question that is asked by the people that live in the future. So, what is the meaning of life?
Life is a lot like a game of poker. It’s a game that you have to play every day. You have to be a good player, you have to know when to bet and


In [ ]:
# Test the text generation - Layerskip
prompt = "What is the meaning of life?"
inputs = layerskip_tokenizer(prompt, return_tensors="pt").to(device)

outputs = layerskip_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True
)
print(layerskip_tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What is the meaning of life? Are we just here to survive? Is there an afterlife? Why do we exist?
These questions are not easy to answer. They are questions that are being explored by philosophers and theologians. It is a question that has been asked for centuries, and it is one that will probably continue to be asked for centuries to come.
The question of life and death is a complex one. It has been asked by philosophers and theologians for centuries, and it is one that will probably continue to be asked


In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
#dataset = load_dataset("tweet_eval", "sentiment")
dataset = load_dataset("tweet_eval", "sentiment")

# Sentiment Analysis

In [ ]:
# Check the available splits
print(dataset)

# See a sample
print(dataset['train'][6])

# Emotions: anger, fear, joy, love, sadness, and surprise
label_names = dataset['train'].features['label'].names
print(label_names)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': '"\\"""" SOUL TRAIN\\"""" OCT 27 HALLOWEEN SPECIAL ft T.dot FINEST rocking the mic...CRAZY CACTUS NIGHT CLUB ..ADV ticket $10 wt out costume $15..."', 'label': 2}
['negative', 'neutral', 'positive']


In [ ]:
index = 10
text = dataset['train'][index]['text']
prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, neutral=1, positive=2): '

# Tokenize input
inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
input_len = inputs.input_ids.shape[1]

# Generate prediction
outputs = base_model.generate(**inputs, max_new_tokens=2)

# Extract the output
generated_text = outputs[0]
generated_ids = generated_text[input_len:]
predicted_text = base_tokenizer.decode(generated_text, skip_special_tokens=True).strip()
predicted = base_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("Output:", predicted_text)
print()
print("Predicted sentiment:", predicted)

print("Actual sentiment:", dataset['train'][index]['label'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output: Please determine the sentiment of the Text.
Text: "@user Well said on HMW. Can you now address why Texans fans file out of the stadium midway through the 4th qtr of every game?"
Sentiment (negative=0, neutral=1, positive=2): 1

Predicted sentiment: 1
Actual sentiment: 1


In [ ]:
base_texts = []
base_raw_predictions = []
base_actuals = []

dataset_type = "test"
d_dataset=len(dataset[dataset_type])
i = 0
for example in dataset[dataset_type]:
  if i % 100 == 0:
    print(f"{i}/{d_dataset}")
  i += 1
  text = example['text']
  true_label = example['label']

  # Build prompt
  prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, neutral=1, positive=2): '

  # Tokenize
  inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
  input_len = inputs.input_ids.shape[1]

  # Generate
  outputs = base_model.generate(**inputs,
                                max_new_tokens=1,
                                pad_token_id=base_tokenizer.eos_token_id)  # silence the warning
  generated_ids = outputs[0][input_len:]

  # Decode
  predicted = base_tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

  base_texts.append(text)
  base_raw_predictions.append(predicted)
  base_actuals.append(true_label)

# Optional: Print first few
for i in range(5):
    print(f"Text: {dataset[dataset_type][i]['text']}")
    print(f"Actual: {base_actuals[i]}")
    print(f"Predicted: {base_raw_predictions[i]}")
    print()

0/12284
100/12284
200/12284
300/12284
400/12284
500/12284
600/12284
700/12284
800/12284
900/12284
1000/12284
1100/12284
1200/12284
1300/12284
1400/12284
1500/12284
1600/12284
1700/12284
1800/12284
1900/12284
2000/12284
2100/12284
2200/12284
2300/12284
2400/12284
2500/12284
2600/12284
2700/12284
2800/12284
2900/12284
3000/12284
3100/12284
3200/12284
3300/12284
3400/12284
3500/12284
3600/12284
3700/12284
3800/12284
3900/12284
4000/12284
4100/12284
4200/12284
4300/12284
4400/12284
4500/12284
4600/12284
4700/12284
4800/12284
4900/12284
5000/12284
5100/12284
5200/12284
5300/12284
5400/12284
5500/12284
5600/12284
5700/12284
5800/12284
5900/12284
6000/12284
6100/12284
6200/12284
6300/12284
6400/12284
6500/12284
6600/12284
6700/12284
6800/12284
6900/12284
7000/12284
7100/12284
7200/12284
7300/12284
7400/12284
7500/12284
7600/12284
7700/12284
7800/12284
7900/12284
8000/12284
8100/12284
8200/12284
8300/12284
8400/12284
8500/12284
8600/12284
8700/12284
8800/12284
8900/12284
9000/12284
9100/12284


In [ ]:
# prompt: calculate the accuracy of the predictions
from sklearn.metrics import accuracy_score, f1_score, classification_report

actuals = base_actuals
raw_predictions = base_raw_predictions
base_predictions = []
predictions = base_predictions

# Convert predictions to integers
for x in raw_predictions:
  if len(x) > 0 and x[0].isnumeric():
    predictions.append(int(x))
  else:
    predictions.append(-1)

# Calculate accuracy
base_accuracy = accuracy_score(actuals, predictions)
print(f"Accuracy: {base_accuracy}")

Accuracy: 0.40247476392054704


In [ ]:
# Map predictions to integer labels
y_pred = base_predictions
y_true = base_actuals  # Assuming these are already integers

# Filter out invalid predictions (e.g., -1 if the model failed)
filtered = [(yt, yp) for yt, yp in zip(y_true, y_pred) if yp != -1]
y_true_filtered, y_pred_filtered = zip(*filtered)

# Calculate F1 score
f1 = f1_score(y_true_filtered, y_pred_filtered, average="macro")

# Optional: Show detailed report
print("Llama 3.2-1B")
print("Accuracy: ", base_accuracy)
print("Macro F1 Score: ", f1)
print(classification_report(y_true_filtered, y_pred_filtered, target_names=["negative", "neutral", "positive"]))

Llama 3.2-1B
Accuracy:  0.40247476392054704
Macro F1 Score:  0.34292694354795467
              precision    recall  f1-score   support

    negative       0.34      0.21      0.26      3971
     neutral       0.49      0.60      0.53      5936
    positive       0.23      0.25      0.24      2374

    accuracy                           0.40     12281
   macro avg       0.35      0.35      0.34     12281
weighted avg       0.39      0.40      0.39     12281



In [ ]:
index = 9
text = dataset['train'][index]['text']
prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, neutral=1, positive=2): '

# Tokenize input
inputs = layerskip_tokenizer(prompt, return_tensors="pt").to(layerskip_model.device)
input_len = inputs.input_ids.shape[1]

# Generate prediction
outputs = layerskip_model.generate(**inputs, max_new_tokens=1)

# Extract the output
generated_text = outputs[0]
generated_ids = generated_text[input_len:]
predicted_text = layerskip_tokenizer.decode(generated_text, skip_special_tokens=True).strip()
predicted = layerskip_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("Output:", predicted_text)
print()
print("Predicted sentiment:", predicted)

print("Actual sentiment:", dataset['train'][index]['label'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output: Please determine the sentiment of the Text.
Text: "@user @user CENA & AJ sitting in a tree K-I-S-S-I-N-G 1st goes AJ's  job then John's cred then goes Vicki with the GM position."
Sentiment (negative=0, neutral=1, positive=2): 1

Predicted sentiment: 1
Actual sentiment: 1


In [ ]:
# Layerskip Model Classification
layerskip_texts = []
layerskip_raw_predictions = []
layerskip_actuals = []
dataset_type = "test"
d_dataset=len(dataset[dataset_type])
i = 0
for example in dataset[dataset_type]:
  if i % 100 == 0:
    print(f"{i}/{d_dataset}")
  i += 1
  text = example['text']
  true_label = example['label']

  # Build prompt
  prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, neutral=1, positive=2): '

  # Tokenize
  inputs = layerskip_tokenizer(prompt, return_tensors="pt").to(layerskip_model.device)
  input_len = inputs.input_ids.shape[1]

  # Generate
  outputs = layerskip_model.generate(**inputs,
                                max_new_tokens=1,
                                pad_token_id=base_tokenizer.eos_token_id)  # silence the warning
  generated_ids = outputs[0][input_len:]

  # Decode
  predicted = layerskip_tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

  layerskip_texts.append(text)
  layerskip_raw_predictions.append(predicted)
  layerskip_actuals.append(true_label)

# Optional: Print first few
for i in range(5):
    print(f"Text: {dataset['test'][i]['text']}")
    print(f"Actual: {layerskip_actuals[i]}")
    print(f"Predicted: {layerskip_raw_predictions[i]}")
    print()

0/12284
100/12284
200/12284
300/12284
400/12284
500/12284
600/12284
700/12284
800/12284
900/12284
1000/12284
1100/12284
1200/12284
1300/12284
1400/12284
1500/12284
1600/12284
1700/12284
1800/12284
1900/12284
2000/12284
2100/12284
2200/12284
2300/12284
2400/12284
2500/12284
2600/12284
2700/12284
2800/12284
2900/12284
3000/12284
3100/12284
3200/12284
3300/12284
3400/12284
3500/12284
3600/12284
3700/12284
3800/12284
3900/12284
4000/12284
4100/12284
4200/12284
4300/12284
4400/12284
4500/12284
4600/12284
4700/12284
4800/12284
4900/12284
5000/12284
5100/12284
5200/12284
5300/12284
5400/12284
5500/12284
5600/12284
5700/12284
5800/12284
5900/12284
6000/12284
6100/12284
6200/12284
6300/12284
6400/12284
6500/12284
6600/12284
6700/12284
6800/12284
6900/12284
7000/12284
7100/12284
7200/12284
7300/12284
7400/12284
7500/12284
7600/12284
7700/12284
7800/12284
7900/12284
8000/12284
8100/12284
8200/12284
8300/12284
8400/12284
8500/12284
8600/12284
8700/12284
8800/12284
8900/12284
9000/12284
9100/12284


In [ ]:
# prompt: calculate the accuracy of the predictions
from sklearn.metrics import accuracy_score, f1_score, classification_report

actuals = layerskip_actuals
raw_predictions = layerskip_raw_predictions
layerskip_predictions = []
predictions = layerskip_predictions

# Convert predictions to integers
for x in raw_predictions:
  if len(x) > 0 and x[0].isnumeric():
    predictions.append(int(x))
  else:
    predictions.append(-1)

# Calculate accuracy
layerskip_accuracy = accuracy_score(actuals, predictions)
print(f"Accuracy: {layerskip_accuracy}")

Accuracy: 0.4735428199283621


In [ ]:
# Map predictions to integer labels
y_pred = layerskip_predictions
y_true = layerskip_actuals  # Assuming these are already integers

# Filter out invalid predictions (e.g., -1 if the model failed)
filtered = [(yt, yp) for yt, yp in zip(y_true, y_pred) if yp != -1]
y_true_filtered, y_pred_filtered = zip(*filtered)

# Calculate F1 score
layerskip_f1 = f1_score(y_true_filtered, y_pred_filtered, average="macro")

# Optional: Show detailed report
print("Layerskip 3.2-1B")
print("Accuracy: ", layerskip_accuracy)
print("Macro F1 Score:", layerskip_f1)
print(classification_report(y_true_filtered, y_pred_filtered, target_names=["negative", "neutral", "positive"]))

Layerskip 3.2-1B
Accuracy:  0.4735428199283621
Macro F1 Score: 0.25740921734415495
              precision    recall  f1-score   support

    negative       0.37      0.09      0.14      3972
     neutral       0.48      0.92      0.63      5937
    positive       0.00      0.00      0.00      2375

    accuracy                           0.47     12284
   macro avg       0.28      0.34      0.26     12284
weighted avg       0.35      0.47      0.35     12284



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Binary Sentiment

In [ ]:
binary_dataset = load_dataset("yelp_polarity")

In [ ]:
# Check the available splits
print(binary_dataset)

# See a sample
print(binary_dataset['train'][6])

# Emotions: anger, fear, joy, love, sadness, and surprise
label_names = binary_dataset['train'].features['label'].names
print(label_names)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 38000
    })
})
{'text': "Owning a driving range inside the city limits is like a license to print money.  I don't think I ask much out of a driving range.  Decent mats, clean balls and accessible hours.  Hell you need even less people now with the advent of the machine that doles out the balls.  This place has none of them.  It is april and there are no grass tees yet.  BTW they opened for the season this week although it has been golfing weather for a month.  The mats look like the carpet at my 107 year old aunt Irene's house.  Worn and thread bare.  Let's talk about the hours.  This place is equipped with lights yet they only sell buckets of balls until 730.  It is still light out.  Finally lets you have the pit to hit into.  When I arrived I wasn't sure if this was a driving range or an excavation site for

In [ ]:
index = 10
text = binary_dataset['train'][index]['text']
prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, positive=1): '

# Tokenize input
inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
input_len = inputs.input_ids.shape[1]

# Generate prediction
outputs = base_model.generate(**inputs, max_new_tokens=2)

# Extract the output
generated_text = outputs[0]
generated_ids = generated_text[input_len:]
predicted_text = base_tokenizer.decode(generated_text, skip_special_tokens=True).strip()
predicted = base_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("Output:", predicted_text)
print()
print("Predicted sentiment:", predicted)

print("Actual sentiment:", binary_dataset['train'][index]['label'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output: Please determine the sentiment of the Text.
Text: "After waiting for almost 30 minutes to trade in an old phone part of the buy back program, our customer service rep incorrectly processed the transaction. This led to us waiting another 30 minutes for him to correct it. Don't visit this store if you want pleasant or good service."
Sentiment (negative=0, positive=1): 0.

Predicted sentiment: 0.
Actual sentiment: 0


In [ ]:
base_bin_texts = []
base_bin_raw_predictions = []
base_bin_actuals = []

dataset_type = "test"
d_dataset=len(binary_dataset[dataset_type])
i = 0
for example in binary_dataset[dataset_type]:
  if i % 100 == 0:
    print(f"{i}/{d_dataset}")

  i += 1
  text = example['text']
  true_label = example['label']

  # Build prompt
  prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, positive=1): '

  # Tokenize
  inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
  input_len = inputs.input_ids.shape[1]

  # Generate
  outputs = base_model.generate(**inputs,
                                max_new_tokens=1,
                                pad_token_id=base_tokenizer.eos_token_id)  # silence the warning
  generated_ids = outputs[0][input_len:]

  # Decode
  predicted = base_tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

  base_bin_texts.append(text)
  base_bin_raw_predictions.append(predicted)
  base_bin_actuals.append(true_label)

# Optional: Print first few
for i in range(5):
    print(f"Text: {binary_dataset[dataset_type][i]['text']}")
    print(f"Actual: {base_bin_actuals[i]}")
    print(f"Predicted: {base_bin_raw_predictions[i]}")
    print()

0/38000
100/38000
200/38000
300/38000
400/38000
500/38000
600/38000
700/38000
800/38000
900/38000
1000/38000
1100/38000
1200/38000
1300/38000
1400/38000
1500/38000
1600/38000
1700/38000
1800/38000
1900/38000
2000/38000
2100/38000
2200/38000
2300/38000
2400/38000
2500/38000
2600/38000
2700/38000
2800/38000
2900/38000
3000/38000
3100/38000
3200/38000
3300/38000
3400/38000
3500/38000
3600/38000
3700/38000
3800/38000
3900/38000
4000/38000
4100/38000
4200/38000
4300/38000
4400/38000
4500/38000
4600/38000
4700/38000
4800/38000
4900/38000
5000/38000
5100/38000
5200/38000
5300/38000
5400/38000
5500/38000
5600/38000
5700/38000
5800/38000
5900/38000
6000/38000
6100/38000
6200/38000
6300/38000
6400/38000
6500/38000
6600/38000
6700/38000
6800/38000
6900/38000
7000/38000
7100/38000
7200/38000
7300/38000
7400/38000
7500/38000
7600/38000
7700/38000
7800/38000
7900/38000
8000/38000
8100/38000
8200/38000
8300/38000
8400/38000
8500/38000
8600/38000
8700/38000
8800/38000
8900/38000
9000/38000
9100/38000


In [ ]:
# prompt: calculate the accuracy of the predictions
from sklearn.metrics import accuracy_score, f1_score, classification_report

actuals = base_bin_actuals
raw_predictions = base_bin_raw_predictions
base_bin_predictions = []
predictions = base_bin_predictions

# Convert predictions to integers
for x in raw_predictions:
  if len(x) > 0 and x[0].isnumeric() and x[0] in ["0", "1"]:
    predictions.append(int(x))
  else:
    predictions.append(-1)

# Calculate accuracy
base_bin_accuracy = accuracy_score(actuals, predictions)
print(f"Accuracy: {base_bin_accuracy}")

Accuracy: 0.5773947368421053


In [ ]:
set(base_bin_predictions + base_bin_actuals)

{-1, 0, 1}

In [ ]:
# Map predictions to integer labels
y_pred = base_bin_predictions
y_true = base_bin_actuals  # Assuming these are already integers

# Filter out invalid predictions (e.g., -1 if the model failed)
filtered = [(yt, yp) for yt, yp in zip(y_true, y_pred) if yp != -1]
y_true_filtered, y_pred_filtered = zip(*filtered)

# Calculate F1 score
base_bin_f1 = f1_score(y_true_filtered, y_pred_filtered, average="macro")

# Optional: Show detailed report
print("Llama 3.2-1B")
print("Accuracy: ", base_bin_accuracy)
print("Macro F1 Score: ", base_bin_f1)
print(classification_report(y_true_filtered, y_pred_filtered, target_names=["negative", "positive"]))

Llama 3.2-1B
Accuracy:  0.5773947368421053
Macro F1 Score:  0.5469379379948953
              precision    recall  f1-score   support

    negative       0.55      0.88      0.68     18749
    positive       0.70      0.29      0.41     18729

    accuracy                           0.59     37478
   macro avg       0.63      0.59      0.55     37478
weighted avg       0.63      0.59      0.55     37478



In [ ]:
index = 9
text = binary_dataset['train'][index]['text']
prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, positive=1): '

# Tokenize input
inputs = layerskip_tokenizer(prompt, return_tensors="pt").to(layerskip_model.device)
input_len = inputs.input_ids.shape[1]

# Generate prediction
outputs = layerskip_model.generate(**inputs, max_new_tokens=1)

# Extract the output
generated_text = outputs[0]
generated_ids = generated_text[input_len:]
predicted_text = layerskip_tokenizer.decode(generated_text, skip_special_tokens=True).strip()
predicted = layerskip_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("Output:", predicted_text)
print()
print("Predicted sentiment:", predicted)

print("Actual sentiment:", binary_dataset['train'][index]['label'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output: Please determine the sentiment of the Text.
Text: "I drove by yesterday to get a sneak peak.  It re-opens on July 14th and I can't wait to take my kids.  The new range looks amazing.  The entire range appears to be turf, which may or many not help your game, but it looks really nice.  The tee boxes look state of the art and the club house looks like something you'll see on a newer course.  Can't wait to experience it!"
Sentiment (negative=0, positive=1): 0

Predicted sentiment: 0
Actual sentiment: 1


In [ ]:
# Layerskip Model Classification
layerskip_bin_texts = []
layerskip_bin_raw_predictions = []
layerskip_bin_actuals = []
dataset_type = "test"
d_dataset=len(binary_dataset[dataset_type])
i = 0
for example in binary_dataset[dataset_type]:
  if i % 100 == 0:
    print(f"{i}/{d_dataset}")
  i += 1
  text = example['text']
  true_label = example['label']

  # Build prompt
  prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, positive=1): '

  # Tokenize
  inputs = layerskip_tokenizer(prompt, return_tensors="pt").to(layerskip_model.device)
  input_len = inputs.input_ids.shape[1]

  # Generate
  outputs = layerskip_model.generate(**inputs,
                                max_new_tokens=1,
                                pad_token_id=base_tokenizer.eos_token_id)  # silence the warning
  generated_ids = outputs[0][input_len:]

  # Decode
  predicted = layerskip_tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

  layerskip_bin_texts.append(text)
  layerskip_bin_raw_predictions.append(predicted)
  layerskip_bin_actuals.append(true_label)

# Optional: Print first few
for i in range(5):
    print(f"Text: {binary_dataset[dataset_type][i]['text']}")
    print(f"Actual: {layerskip_bin_actuals[i]}")
    print(f"Predicted: {layerskip_bin_raw_predictions[i]}")
    print()

0/38000
100/38000
200/38000
300/38000
400/38000
500/38000
600/38000
700/38000
800/38000
900/38000
1000/38000
1100/38000
1200/38000
1300/38000
1400/38000
1500/38000
1600/38000
1700/38000
1800/38000
1900/38000
2000/38000
2100/38000
2200/38000
2300/38000
2400/38000
2500/38000
2600/38000
2700/38000
2800/38000
2900/38000
3000/38000
3100/38000
3200/38000
3300/38000
3400/38000
3500/38000
3600/38000
3700/38000
3800/38000
3900/38000
4000/38000
4100/38000
4200/38000
4300/38000
4400/38000
4500/38000
4600/38000
4700/38000
4800/38000
4900/38000
5000/38000
5100/38000
5200/38000
5300/38000
5400/38000
5500/38000
5600/38000
5700/38000
5800/38000
5900/38000
6000/38000
6100/38000
6200/38000
6300/38000
6400/38000
6500/38000
6600/38000
6700/38000
6800/38000
6900/38000
7000/38000
7100/38000
7200/38000
7300/38000
7400/38000
7500/38000
7600/38000
7700/38000
7800/38000
7900/38000
8000/38000
8100/38000
8200/38000
8300/38000
8400/38000
8500/38000
8600/38000
8700/38000
8800/38000
8900/38000
9000/38000
9100/38000


In [ ]:
# prompt: calculate the accuracy of the predictions
from sklearn.metrics import accuracy_score, f1_score, classification_report

actuals = layerskip_bin_actuals
raw_predictions = layerskip_bin_raw_predictions
layerskip_bin_predictions = []
predictions = layerskip_bin_predictions

# Convert predictions to integers
for x in raw_predictions:
  if len(x) > 0 and x[0].isnumeric():
    predictions.append(int(x))
  else:
    predictions.append(-1)

# Calculate accuracy
layerskip_bin_accuracy = accuracy_score(actuals, predictions)
print(f"Accuracy: {layerskip_bin_accuracy}")

Accuracy: 0.5002105263157894


In [ ]:
# Map predictions to integer labels
y_pred = layerskip_bin_predictions
y_true = layerskip_bin_actuals  # Assuming these are already integers

# Filter out invalid predictions (e.g., -1 if the model failed)
filtered = [(yt, yp) for yt, yp in zip(y_true, y_pred) if yp != -1]
y_true_filtered, y_pred_filtered = zip(*filtered)

# Calculate F1 score
layerskip_bin_f1 = f1_score(y_true_filtered, y_pred_filtered, average="macro")

# Optional: Show detailed report
print("Layerskip 3.2-1B")
print("Accuracy: ", layerskip_bin_accuracy)
print("Macro F1 Score:", layerskip_bin_f1)
print(classification_report(y_true_filtered, y_pred_filtered, target_names=["negative", "positive"]))

Layerskip 3.2-1B
Accuracy:  0.5002105263157894
Macro F1 Score: 0.33380099894698434
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67     19000
    positive       1.00      0.00      0.00     19000

    accuracy                           0.50     38000
   macro avg       0.75      0.50      0.33     38000
weighted avg       0.75      0.50      0.33     38000



# Binary Sentiment Analysis: SST-2 Dataset

In [ ]:
sst_dataset = load_dataset("glue", "sst2")

In [ ]:
# Check the available splits
print(sst_dataset)

# See a sample
print(sst_dataset['train'][6])

# Emotions: anger, fear, joy, love, sadness, and surprise
label_names = sst_dataset['train'].features['label'].names
print(label_names)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
{'sentence': 'demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . ', 'label': 1, 'idx': 6}
['negative', 'positive']


In [ ]:
index = 4
text = sst_dataset['train'][index]['sentence']
prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, positive=1): '

# Tokenize input
inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
input_len = inputs.input_ids.shape[1]

# Generate prediction
outputs = base_model.generate(**inputs, max_new_tokens=2)

# Extract the output
generated_text = outputs[0]
generated_ids = generated_text[input_len:]
predicted_text = base_tokenizer.decode(generated_text, skip_special_tokens=True).strip()
predicted = base_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("Output:", predicted_text)
print()
print("Predicted sentiment:", predicted)

print("Actual sentiment:", sst_dataset['train'][index]['label'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output: Please determine the sentiment of the Text.
Text: "on the worst revenge-of-the-nerds clichés the filmmakers could dredge up "
Sentiment (negative=0, positive=1): 0.

Predicted sentiment: 0.
Actual sentiment: 0


In [ ]:
base_sst_texts = []
base_sst_raw_predictions = []
base_sst_actuals = []

dataset_type = "train"
d_dataset=len(sst_dataset[dataset_type])
i = 0
for example in sst_dataset[dataset_type]:
  if i % 100 == 0:
    print(f"{i}/{d_dataset}")

  i += 1
  text = example['sentence']
  true_label = example['label']

  # Build prompt
  prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, positive=1): '

  # Tokenize
  inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
  input_len = inputs.input_ids.shape[1]

  # Generate
  outputs = base_model.generate(**inputs,
                                max_new_tokens=1,
                                pad_token_id=base_tokenizer.eos_token_id)  # silence the warning
  generated_ids = outputs[0][input_len:]

  # Decode
  predicted = base_tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

  base_sst_texts.append(text)
  base_sst_raw_predictions.append(predicted)
  base_sst_actuals.append(true_label)

# Optional: Print first few
for i in range(5):
    print(f"Text: {sst_dataset[dataset_type][i]['sentence']}")
    print(f"Actual: {base_sst_actuals[i]}")
    print(f"Predicted: {base_sst_raw_predictions[i]}")
    print()

0/67349
100/67349
200/67349
300/67349
400/67349
500/67349
600/67349
700/67349
800/67349
900/67349
1000/67349
1100/67349
1200/67349
1300/67349
1400/67349
1500/67349
1600/67349
1700/67349
1800/67349
1900/67349
2000/67349
2100/67349
2200/67349
2300/67349
2400/67349
2500/67349
2600/67349
2700/67349
2800/67349
2900/67349
3000/67349
3100/67349
3200/67349
3300/67349
3400/67349
3500/67349
3600/67349
3700/67349
3800/67349
3900/67349
4000/67349
4100/67349
4200/67349
4300/67349
4400/67349
4500/67349
4600/67349
4700/67349
4800/67349
4900/67349
5000/67349
5100/67349
5200/67349
5300/67349
5400/67349
5500/67349
5600/67349
5700/67349
5800/67349
5900/67349
6000/67349
6100/67349
6200/67349
6300/67349
6400/67349
6500/67349
6600/67349
6700/67349
6800/67349
6900/67349
7000/67349
7100/67349
7200/67349
7300/67349
7400/67349
7500/67349
7600/67349
7700/67349
7800/67349
7900/67349
8000/67349
8100/67349
8200/67349
8300/67349
8400/67349
8500/67349
8600/67349
8700/67349
8800/67349
8900/67349
9000/67349
9100/67349


KeyboardInterrupt: 

In [ ]:
# prompt: calculate the accuracy of the predictions
from sklearn.metrics import accuracy_score, f1_score, classification_report

actuals = base_sst_actuals
raw_predictions = base_sst_raw_predictions
base_sst_predictions = []
predictions = base_sst_predictions

# Convert predictions to integers
for x in raw_predictions:
  if len(x) > 0 and x[0].isnumeric():
    predictions.append(int(x))
  else:
    predictions.append(-1)

# Calculate accuracy
base_sst_accuracy = accuracy_score(actuals, predictions)
print(f"Accuracy: {base_sst_accuracy}")

Accuracy: 0.48398977751052313


In [ ]:
# Map predictions to integer labels
y_pred = base_sst_predictions
y_true = base_sst_actuals  # Assuming these are already integers

# Filter out invalid predictions (e.g., -1 if the model failed)
filtered = [(yt, yp) for yt, yp in zip(y_true, y_pred) if yp != -1]
y_true_filtered, y_pred_filtered = zip(*filtered)

# Calculate F1 score
base_sst_f1 = f1_score(y_true_filtered, y_pred_filtered, average="macro")

# Optional: Show detailed report
print("Llama 3.2-1B")
print("Accuracy: ", base_sst_accuracy)
print("Macro F1 Score: ", base_sst_f1)
print(classification_report(y_true_filtered, y_pred_filtered, target_names=["negative", "positive"]))

Llama 3.2-1B
Accuracy:  0.48398977751052313
Macro F1 Score:  0.44885636529327555
              precision    recall  f1-score   support

    negative       0.45      0.83      0.59     17618
    positive       0.61      0.21      0.31     22294

    accuracy                           0.48     39912
   macro avg       0.53      0.52      0.45     39912
weighted avg       0.54      0.48      0.43     39912



In [ ]:
index = 9
text = sst_dataset['train'][index]['sentence']
prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, positive=1): '

# Tokenize input
inputs = layerskip_tokenizer(prompt, return_tensors="pt").to(layerskip_model.device)
input_len = inputs.input_ids.shape[1]

# Generate prediction
outputs = layerskip_model.generate(**inputs, max_new_tokens=1)

# Extract the output
generated_text = outputs[0]
generated_ids = generated_text[input_len:]
predicted_text = layerskip_tokenizer.decode(generated_text, skip_special_tokens=True).strip()
predicted = layerskip_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("Output:", predicted_text)
print()
print("Predicted sentiment:", predicted)

print("Actual sentiment:", sst_dataset['train'][index]['label'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output: Please determine the sentiment of the Text.
Text: "are more deeply thought through than in most ` right-thinking'films "
Sentiment (negative=0, positive=1): 0

Predicted sentiment: 0
Actual sentiment: 1


In [ ]:
# Layerskip Model Classification
layerskip_sst_texts = []
layerskip_sst_raw_predictions = []
layerskip_sst_actuals = []
dataset_type = "train"
d_dataset=len(sst_dataset[dataset_type])
i = 0
for example in sst_dataset[dataset_type]:
  if i % 100 == 0:
    print(f"{i}/{d_dataset}")
  i += 1
  text = example['sentence']
  true_label = example['label']

  # Build prompt
  prompt = f'Please determine the sentiment of the Text.\nText: "{text}"\nSentiment (negative=0, positive=1): '

  # Tokenize
  inputs = layerskip_tokenizer(prompt, return_tensors="pt").to(layerskip_model.device)
  input_len = inputs.input_ids.shape[1]

  # Generate
  outputs = layerskip_model.generate(**inputs,
                                max_new_tokens=1,
                                pad_token_id=base_tokenizer.eos_token_id)  # silence the warning
  generated_ids = outputs[0][input_len:]

  # Decode
  predicted = layerskip_tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

  layerskip_sst_texts.append(text)
  layerskip_sst_raw_predictions.append(predicted)
  layerskip_sst_actuals.append(true_label)

# Optional: Print first few
for i in range(5):
    print(f"Text: {sst_dataset[dataset_type][i]['sentence']}")
    print(f"Actual: {layerskip_sst_actuals[i]}")
    print(f"Predicted: {layerskip_sst_raw_predictions[i]}")
    print()

0/67349
100/67349
200/67349
300/67349
400/67349
500/67349
600/67349
700/67349
800/67349
900/67349
1000/67349
1100/67349
1200/67349
1300/67349
1400/67349
1500/67349
1600/67349
1700/67349
1800/67349
1900/67349
2000/67349
2100/67349
2200/67349
2300/67349
2400/67349
2500/67349
2600/67349
2700/67349
2800/67349
2900/67349
3000/67349
3100/67349
3200/67349
3300/67349
3400/67349
3500/67349
3600/67349
3700/67349
3800/67349
3900/67349
4000/67349
4100/67349
4200/67349
4300/67349
4400/67349
4500/67349
4600/67349
4700/67349
4800/67349
4900/67349
5000/67349
5100/67349
5200/67349
5300/67349
5400/67349
5500/67349
5600/67349
5700/67349
5800/67349
5900/67349
6000/67349
6100/67349
6200/67349
6300/67349
6400/67349
6500/67349
6600/67349
6700/67349
6800/67349
6900/67349
7000/67349
7100/67349
7200/67349
7300/67349
7400/67349
7500/67349
7600/67349
7700/67349
7800/67349
7900/67349
8000/67349
8100/67349
8200/67349
8300/67349
8400/67349
8500/67349
8600/67349
8700/67349
8800/67349
8900/67349
9000/67349
9100/67349


KeyboardInterrupt: 

In [ ]:
# prompt: calculate the accuracy of the predictions
from sklearn.metrics import accuracy_score, f1_score, classification_report

actuals = layerskip_sst_actuals
raw_predictions = layerskip_sst_raw_predictions
layerskip_sst_predictions = []
predictions = layerskip_sst_predictions

# Convert predictions to integers
for x in raw_predictions:
  if len(x) > 0 and x[0].isnumeric():
    predictions.append(int(x))
  else:
    predictions.append(-1)

# Calculate accuracy
layerskip_sst_accuracy = accuracy_score(actuals, predictions)
print(f"Accuracy: {layerskip_sst_accuracy}")

Accuracy: 0.44911115452236766


In [ ]:
# Map predictions to integer labels
y_pred = layerskip_sst_predictions
y_true = layerskip_sst_actuals  # Assuming these are already integers

# Filter out invalid predictions (e.g., -1 if the model failed)
filtered = [(yt, yp) for yt, yp in zip(y_true, y_pred) if yp != -1]
y_true_filtered, y_pred_filtered = zip(*filtered)

# Calculate F1 score
layerskip_sst_f1 = f1_score(y_true_filtered, y_pred_filtered, average="macro")

# Optional: Show detailed report
print("Layerskip 3.2-1B")
print("Accuracy: ", layerskip_sst_accuracy)
print("Macro F1 Score:", layerskip_sst_f1)
print(classification_report(y_true_filtered, y_pred_filtered, target_names=["negative", "positive"]))

Layerskip 3.2-1B
Accuracy:  0.44911115452236766
Macro F1 Score: 0.309921811809113
              precision    recall  f1-score   support

    negative       0.45      1.00      0.62      4598
    positive       0.00      0.00      0.00      5640

    accuracy                           0.45     10238
   macro avg       0.22      0.50      0.31     10238
weighted avg       0.20      0.45      0.28     10238



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
